# 유지율 분석 (Retention)

- 기초 지표를 확인한 결과, 서비스 이용 과정에서 **사용자의 지속적인 이용이 급격하게 감소하는 패턴**이 확인되었으며, 이를 바탕으로 **Retention을 주요 문제 영역으로 정의**했습니다.

- 따라서 본 분석에서는 사용자의 지속적인 서비스 이용에 영향을 미칠 수 있는 요인에 대해 **가설을 설정하고 데이터를 통해 검증**한 뒤, 이를 바탕으로 **Retention 개선을 위한 서비스 개선안을 도출하는 것**을 목표로 합니다.

### Retention 기준

- 본 서비스에서 사용자의 지속적인 이용 여부를 판단하기 위한 **핵심 행동을 `질문셋 완료`로 정의**합니다.
- 질문셋 완료 여부는 `polls_questionset` 테이블의 `status = 'F'`를 기준으로 판단합니다.

### 기능 이용 가능 시점

- 사용자가 질문 기능을 이용하기 위해서는 아래 조건을 충족해야 합니다.
  - **학교 인원 40명 이상**
  - **친구 인원 4명 이상**

- 따라서 단순 가입 시점을 Retention의 시작점으로 설정하지 않고, **두 조건을 모두 충족하여 실제로 핵심 기능을 이용할 수 있게 된 시점**을 기준으로 이후 사용자의 질문셋 완료 및 지속 이용 여부를 확인합니다.

## 0. 라이브러리 호출 및 환경 세팅

### 0-1. 라이브러리 호출

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from google.cloud import bigquery

### 0-2. 시각화 공통 환경 세팅

In [2]:
sns.set_theme(style="darkgrid")

plt.rc('font', family='AppleGothic')
# 마이너스 폰트 깨짐 방지 설정
plt.rcParams['axes.unicode_minus'] = False

### 0-3. 함수 정의

### 0-4. 데이터 호출 및 준비

In [3]:
PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

#### 0-4-1. 타깃 학교 정의

In [5]:
target_school_ids = (
    271, 352, 369, 1478, 1719,
    4426, 4516, 5372, 5491, 5520,
)

#### 0-4-2. 타깃 유저 추출

In [ ]:
target_user_sql = f"""
SELECT
    u.id AS user_id,
    u.created_at AS signup_at,
    u.group_id,
    g.school_id
FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
    ON u.group_id = g.id
WHERE g.school_id IN {target_school_ids}
"""

target_users = (
    client.query(target_user_sql)
    .to_dataframe()
)

#### 0-4-3. 타깃 유저 데이터 검정

In [8]:
display(target_users.head())

print(f'행 수: {len(target_users):,}')
print(f'고유 사용자 수: {target_users["user_id"].nunique():,}')
print(f'학교 수: {target_users["school_id"].nunique():,}')

,user_id,signup_at,group_id,school_id
0,1116196,2023-05-11 02:55:31.224481+00:00,12343,352
1,1007303,2023-05-08 12:03:56.593162+00:00,10276,4516
2,870224,2023-05-02 09:34:43.446143+00:00,6513,369
3,888666,2023-05-05 14:58:46.649592+00:00,4286,271
4,884682,2023-05-05 08:27:51.933459+00:00,9418,4516


행 수: 5,090
고유 사용자 수: 5,090
학교 수: 10


#### 0-4-4. 학교별 활성화 시간 산출

- 가입 데이터를 기준으로 해당 학교 소속의 유저가 40명이 되는 시점의 날짜 및 시간을 체크하여 해당 학교의 활성화 시간을 산출합니다.

In [17]:
school_signup_order = (
    target_users
    .sort_values(
        ['school_id', 'signup_at', 'user_id']
    )
    .copy()
)

school_signup_order['school_signup_order'] = (
    school_signup_order
    .groupby('school_id')
    .cumcount()
    + 1
)

In [18]:
display(
    school_signup_order[
        [
            'school_id',
            'user_id',
            'signup_at',
            'school_signup_order',
        ]
    ]
    .head()
)

,school_id,user_id,signup_at,school_signup_order
1911,271,838023,2023-04-19 09:06:00.719792+00:00,1
1591,271,838642,2023-04-20 00:11:00.962160+00:00,2
4137,271,839357,2023-04-20 14:26:20.479856+00:00,3
950,271,839670,2023-04-21 00:25:27.279382+00:00,4
880,271,845238,2023-04-24 22:15:16.785006+00:00,5


In [19]:
school_40_at = (
    school_signup_order[
        school_signup_order['school_signup_order'].eq(40)
    ]
    [
        [
            'school_id',
            'signup_at',
        ]
    ]
    .rename(
        columns={
            'signup_at': 'school_40_at',
        }
    )
    .reset_index(drop=True)
)

display(school_40_at)

,school_id,school_40_at
0,271,2023-04-28 03:43:18.503902+00:00
1,352,2023-05-05 04:07:00.053881+00:00
2,369,2023-05-02 22:44:27.267798+00:00
3,1478,2023-05-06 14:18:58.057918+00:00
4,1719,2023-05-13 08:00:50.712386+00:00
5,4426,2023-05-16 00:18:41.426463+00:00
6,4516,2023-05-02 15:17:04.497555+00:00
7,5372,2023-05-10 23:47:35.094548+00:00
8,5491,2023-05-07 02:57:28.039304+00:00
9,5520,2023-05-08 13:49:27.604223+00:00


In [20]:
print(f'타깃 학교 수: {target_users["school_id"].nunique():,}개')
print(f'40명 도달 학교 수: {school_40_at["school_id"].nunique():,}개')

타깃 학교 수: 10개
40명 도달 학교 수: 10개


In [21]:
school_user_counts = (
    target_users
    .groupby('school_id')['user_id']
    .nunique()
    .reset_index(name='가입자 수')
)

school_40_check = (
    school_user_counts
    .merge(
        school_40_at,
        on='school_id',
        how='left',
        validate='one_to_one',
    )
)

display(school_40_check)

,school_id,가입자 수,school_40_at
0,271,492,2023-04-28 03:43:18.503902+00:00
1,352,491,2023-05-05 04:07:00.053881+00:00
2,369,578,2023-05-02 22:44:27.267798+00:00
3,1478,488,2023-05-06 14:18:58.057918+00:00
4,1719,551,2023-05-13 08:00:50.712386+00:00
5,4426,487,2023-05-16 00:18:41.426463+00:00
6,4516,510,2023-05-02 15:17:04.497555+00:00
7,5372,507,2023-05-10 23:47:35.094548+00:00
8,5491,486,2023-05-07 02:57:28.039304+00:00
9,5520,500,2023-05-08 13:49:27.604223+00:00


#### 0-4-5. 유저별 활성화 시간 산출

In [28]:
target_friend_history_sql = f"""
WITH target_users AS (
    SELECT
        u.id AS user_id,
        g.school_id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
        ON u.group_id = g.id
    WHERE g.school_id IN {target_school_ids}
)

SELECT
    f.user_id,
    f.friendship_at,
    f.friend_cnt_after
FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history` AS f
INNER JOIN target_users AS u
    ON f.user_id = u.user_id
ORDER BY
    f.user_id,
    f.friendship_at,
    f.friend_user_id
"""

target_friend_history = (
    client.query(target_friend_history_sql)
    .to_dataframe()
)

In [29]:
display(target_friend_history.head())

print(
    f'친구 관계 로그 수: '
    f'{len(target_friend_history):,}건'
)

print(
    f'친구 관계가 있는 타깃 사용자 수: '
    f'{target_friend_history["user_id"].nunique():,}명'
)

print(
    f'전체 타깃 사용자 수: '
    f'{target_users["user_id"].nunique():,}명'
)

,user_id,friendship_at,friend_cnt_after
0,838466,2023-05-01 14:10:10+00:00,1
1,838466,2023-05-01 14:10:11+00:00,2
2,838466,2023-05-01 14:10:14+00:00,3
3,838466,2023-05-01 14:10:16+00:00,4
4,838466,2023-05-01 14:10:18+00:00,5


친구 관계 로그 수: 221,675건
친구 관계가 있는 타깃 사용자 수: 4,980명
전체 타깃 사용자 수: 5,090명


In [30]:
user_4th_friend_at = (
    target_friend_history[
        target_friend_history['friend_cnt_after'].eq(4)
    ]
    [
        [
            'user_id',
            'friendship_at',
        ]
    ]
    .rename(
        columns={
            'friendship_at': 'friend_4_at',
        }
    )
    .reset_index(drop=True)
)

In [31]:
display(user_4th_friend_at.head())

print(
    f'친구 4명에 도달한 사용자 수: '
    f'{user_4th_friend_at["user_id"].nunique():,}명'
)

print(
    f'결과 행 수: '
    f'{len(user_4th_friend_at):,}개'
)

,user_id,friend_4_at
0,838466,2023-05-01 14:10:16+00:00
1,838642,2023-05-05 07:21:24+00:00
2,839357,2023-04-29 16:55:27+00:00
3,840293,2023-05-04 11:29:51+00:00
4,840473,2023-05-04 13:50:41+00:00


친구 4명에 도달한 사용자 수: 4,880명
결과 행 수: 4,880개


In [32]:
print(
    '사용자 ID 중복 수:',
    user_4th_friend_at['user_id'].duplicated().sum(),
)

사용자 ID 중복 수: 0


#### 0-4-6. 앱 사용이 가능한 유저 데이터

In [33]:
user_activation = (
    target_users
    .merge(
        school_40_at,
        on='school_id',
        how='left',
        validate='many_to_one',
    )
    .merge(
        user_4th_friend_at,
        on='user_id',
        how='left',
        validate='one_to_one',
    )
)

In [34]:
display(user_activation.head())

,user_id,signup_at,group_id,school_id,school_40_at,friend_4_at
0,1116196,2023-05-11 02:55:31.224481+00:00,12343,352,2023-05-05 04:07:00.053881+00:00,2023-05-11 03:11:37+00:00
1,1007303,2023-05-08 12:03:56.593162+00:00,10276,4516,2023-05-02 15:17:04.497555+00:00,2023-05-08 13:35:55+00:00
2,870224,2023-05-02 09:34:43.446143+00:00,6513,369,2023-05-02 22:44:27.267798+00:00,2023-05-02 22:42:20+00:00
3,888666,2023-05-05 14:58:46.649592+00:00,4286,271,2023-04-28 03:43:18.503902+00:00,NaT
4,884682,2023-05-05 08:27:51.933459+00:00,9418,4516,2023-05-02 15:17:04.497555+00:00,2023-05-05 09:31:02+00:00


In [35]:
user_activation['activation_eligible'] = (
    user_activation['school_40_at'].notna()
    & user_activation['friend_4_at'].notna()
)

In [36]:
user_activation['activation_at'] = (
    user_activation[
        [
            'signup_at',
            'school_40_at',
            'friend_4_at',
        ]
    ]
    .max(axis=1)
)

# 두 활성화 조건을 모두 충족하지 못한 사용자는 활성화 시각 없음
user_activation.loc[
    ~user_activation['activation_eligible'],
    'activation_at',
] = pd.NaT

In [39]:
display(
    user_activation[
        [
            'user_id',
            'school_id',
            'signup_at',
            'school_40_at',
            'friend_4_at',
            'activation_eligible',
            'activation_at',
        ]
    ]
    .head()
)

,user_id,school_id,signup_at,school_40_at,friend_4_at,activation_eligible,activation_at
0,1116196,352,2023-05-11 02:55:31.224481+00:00,2023-05-05 04:07:00.053881+00:00,2023-05-11 03:11:37+00:00,True,2023-05-11 03:11:37+00:00
1,1007303,4516,2023-05-08 12:03:56.593162+00:00,2023-05-02 15:17:04.497555+00:00,2023-05-08 13:35:55+00:00,True,2023-05-08 13:35:55+00:00
2,870224,369,2023-05-02 09:34:43.446143+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-02 22:42:20+00:00,True,2023-05-02 22:44:27.267798+00:00
3,888666,271,2023-05-05 14:58:46.649592+00:00,2023-04-28 03:43:18.503902+00:00,NaT,False,NaT
4,884682,4516,2023-05-05 08:27:51.933459+00:00,2023-05-02 15:17:04.497555+00:00,2023-05-05 09:31:02+00:00,True,2023-05-05 09:31:02+00:00


In [38]:
total_users = user_activation['user_id'].nunique()

activated_users = (
    user_activation['activation_eligible'].sum()
)

activation_rate = (
    activated_users / total_users * 100
)

print(f'전체 타깃 사용자 수: {total_users:,}명')
print(f'활성화 가능 사용자 수: {activated_users:,}명')
print(f'활성화 가능 사용자 비율: {activation_rate:.2f}%')

전체 타깃 사용자 수: 5,090명
활성화 가능 사용자 수: 4,880명
활성화 가능 사용자 비율: 95.87%


#### 0-4-7. 질문셋 완료 시각 대리값 산출

- `polls_questionset.status = 'F'`를 질문셋 완료 여부로 사용합니다.
- 별도의 완료 시각이 없으므로, 질문셋에 포함된 질문 조각별 응답 중 가장 늦은 `answer_updated_at`을 `estimated_completed_at`으로 사용합니다.
- 분석 대상 데이터에서 동일한 `user_id + question_piece_id` 조합의 반복이 확인되지 않았으므로 두 컬럼을 기준으로 연결합니다.
- 질문 스킵은 응답 기록을 남기지 않으므로, 연결된 응답 개수와 관계없이 `opening_time` 이후 확인되는 가장 늦은 응답 시각을 사용합니다.
- `estimated_completed_at`은 실제 완료 버튼 클릭 시각이 아니라 마지막 응답 시각을 이용한 대리값입니다.


In [46]:
target_questionset_sql = f"""
WITH target_users AS (
    SELECT
        u.id AS user_id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
        ON u.group_id = g.id
    WHERE g.school_id IN {target_school_ids}
),

finished_questionsets AS (
    SELECT
        qs.id AS questionset_id,
        qs.user_id,
        qs.opening_time,
        qs.question_piece_id_list
    FROM `{PROJECT_ID}.{DATA_SET}.polls_questionset` AS qs
    INNER JOIN target_users AS u
        ON qs.user_id = u.user_id
    WHERE qs.status = 'F'
),

questionset_pieces AS (
    SELECT
        qs.questionset_id,
        qs.user_id,
        qs.opening_time,
        SAFE_CAST(question_piece_id AS INT64) AS question_piece_id
    FROM finished_questionsets AS qs
    CROSS JOIN UNNEST(
        JSON_VALUE_ARRAY(qs.question_piece_id_list)
    ) AS question_piece_id
)

SELECT
    qp.questionset_id,
    qp.user_id,
    qp.opening_time,
    COUNT(*) AS question_piece_count,
    COUNTIF(uqr.id IS NOT NULL) AS matched_record_count,
    COUNTIF(uqr.answer_updated_at IS NOT NULL) AS answered_piece_count,
    COUNTIF(uqr.answer_updated_at < qp.opening_time) AS answer_before_open_count,
    MAX(
        IF(
            uqr.answer_updated_at >= qp.opening_time,
            uqr.answer_updated_at,
            NULL
        )
    ) AS estimated_completed_at
FROM questionset_pieces AS qp
LEFT JOIN `{PROJECT_ID}.{DATA_SET}.accounts_userquestionrecord` AS uqr
    ON qp.user_id = uqr.user_id
    AND qp.question_piece_id = uqr.question_piece_id
GROUP BY
    qp.questionset_id,
    qp.user_id,
    qp.opening_time
ORDER BY
    qp.user_id,
    qp.opening_time
"""

target_questionsets = (
    client.query(target_questionset_sql)
    .to_dataframe()
)


In [47]:
target_questionsets['completion_time_usable'] = (
    target_questionsets['estimated_completed_at'].notna()
    & target_questionsets['answer_before_open_count'].eq(0)
)

display(target_questionsets.head())

print(f'완료 질문셋 수: {len(target_questionsets):,}개')
print(
    '대리 완료 시각을 사용할 수 있는 질문셋 수: '
    f'{target_questionsets["completion_time_usable"].sum():,}개'
)
print(
    '대리 완료 시각 사용 가능 비율: '
    f'{target_questionsets["completion_time_usable"].mean() * 100:.2f}%'
)


,questionset_id,user_id,opening_time,question_piece_count,matched_record_count,answered_piece_count,answer_before_open_count,estimated_completed_at,completion_time_usable
0,116704,838023,2023-04-29 10:02:26+00:00,10,1,1,0,2023-04-29 16:22:56+00:00,True
1,132741,838023,2023-04-29 17:13:06+00:00,10,6,6,0,2023-04-30 04:18:56+00:00,True
2,145316,838023,2023-04-30 05:09:03+00:00,10,1,1,0,2023-05-02 05:49:35+00:00,True
3,257385,838023,2023-05-02 06:39:51+00:00,10,3,3,0,2023-05-03 08:36:09+00:00,True
4,333471,838023,2023-05-03 09:26:09+00:00,10,7,7,0,2023-05-03 12:23:17+00:00,True


완료 질문셋 수: 153,411개
대리 완료 시각을 사용할 수 있는 질문셋 수: 151,562개
대리 완료 시각 사용 가능 비율: 98.79%


In [48]:
questionset_completion_check = pd.Series({
    '일부 질문의 응답 기록이 없는 질문셋(스킵 포함)': (
        target_questionsets['matched_record_count']
        < target_questionsets['question_piece_count']
    ).sum(),
    '일부 질문의 응답 시각이 없는 질문셋(스킵 포함)': (
        target_questionsets['answered_piece_count']
        < target_questionsets['question_piece_count']
    ).sum(),
    '오픈 이전 응답 포함 질문셋': (
        target_questionsets['answer_before_open_count'] > 0
    ).sum(),
    '대리 완료 시각 미산출 질문셋': (
        target_questionsets['estimated_completed_at'].isna()
    ).sum(),
}, name='질문셋 수').to_frame()

display(questionset_completion_check)


,질문셋 수
일부 질문의 응답 기록이 없는 질문셋(스킵 포함),101253
일부 질문의 응답 시각이 없는 질문셋(스킵 포함),101253
오픈 이전 응답 포함 질문셋,4
대리 완료 시각 미산출 질문셋,1845


In [54]:
questionset_completion = (
    target_questionsets[
        target_questionsets['completion_time_usable']
    ]
    [[
        'questionset_id',
        'user_id',
        'opening_time',
        'estimated_completed_at',
    ]]
    .copy()
)

display(questionset_completion.head())


,questionset_id,user_id,opening_time,estimated_completed_at
0,116704,838023,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00
1,132741,838023,2023-04-29 17:13:06+00:00,2023-04-30 04:18:56+00:00
2,145316,838023,2023-04-30 05:09:03+00:00,2023-05-02 05:49:35+00:00
3,257385,838023,2023-05-02 06:39:51+00:00,2023-05-03 08:36:09+00:00
4,333471,838023,2023-05-03 09:26:09+00:00,2023-05-03 12:23:17+00:00


In [55]:
target_questionsets.assign(
    unusable_reason=np.select(
        [
            target_questionsets['estimated_completed_at'].isna(),
            target_questionsets['answer_before_open_count'].gt(0),
        ],
        [
            '오픈 이후 응답 시각 없음',
            '오픈 이전 응답 포함',
        ],
        default='사용 가능',
    )
)['unusable_reason'].value_counts(dropna=False)

unusable_reason
사용 가능             151562
오픈 이후 응답 시각 없음      1845
오픈 이전 응답 포함            4
Name: count, dtype: int64

#### 0-4-8. 활성화된 유저의 질문 세트 완료 기록 연결

In [60]:
questionset_completion

,questionset_id,user_id,opening_time,estimated_completed_at
0,116704,838023,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00
1,132741,838023,2023-04-29 17:13:06+00:00,2023-04-30 04:18:56+00:00
2,145316,838023,2023-04-30 05:09:03+00:00,2023-05-02 05:49:35+00:00
3,257385,838023,2023-05-02 06:39:51+00:00,2023-05-03 08:36:09+00:00
4,333471,838023,2023-05-03 09:26:09+00:00,2023-05-03 12:23:17+00:00
...,...,...,...,...
153406,20592064,1579418,2023-07-26 04:35:44+00:00,2023-07-26 15:20:02+00:00
153407,20595806,1579418,2023-07-26 16:00:03+00:00,2023-07-28 12:16:05+00:00
153408,20608740,1579418,2023-07-28 12:56:06+00:00,2023-07-29 15:29:52+00:00
153409,20617176,1579418,2023-07-29 16:09:52+00:00,2023-08-05 13:19:15+00:00


In [56]:
user_questionset_activity = (
    questionset_completion
    .merge(
        user_activation[
            [
                'user_id',
                'school_id',
                'activation_at',
                'activation_eligible',
            ]
        ],
        on='user_id',
        how='inner',
        validate='many_to_one',
    )
)

In [57]:
display(user_questionset_activity.head())

print(f'전체 행 수: {len(user_questionset_activity):,}')
print(
    '질문셋 완료 경험 사용자 수:',
    f'{user_questionset_activity["user_id"].nunique():,}',
)

,questionset_id,user_id,opening_time,estimated_completed_at,school_id,activation_at,activation_eligible
0,116704,838023,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00,271,NaT,False
1,132741,838023,2023-04-29 17:13:06+00:00,2023-04-30 04:18:56+00:00,271,NaT,False
2,145316,838023,2023-04-30 05:09:03+00:00,2023-05-02 05:49:35+00:00,271,NaT,False
3,257385,838023,2023-05-02 06:39:51+00:00,2023-05-03 08:36:09+00:00,271,NaT,False
4,333471,838023,2023-05-03 09:26:09+00:00,2023-05-03 12:23:17+00:00,271,NaT,False


전체 행 수: 151,562
질문셋 완료 경험 사용자 수: 4,812


In [59]:
user_questionset_activity[user_questionset_activity['activation_eligible'] == False]

,questionset_id,user_id,opening_time,estimated_completed_at,school_id,activation_at,activation_eligible
0,116704,838023,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00,271,NaT,False
1,132741,838023,2023-04-29 17:13:06+00:00,2023-04-30 04:18:56+00:00,271,NaT,False
2,145316,838023,2023-04-30 05:09:03+00:00,2023-05-02 05:49:35+00:00,271,NaT,False
3,257385,838023,2023-05-02 06:39:51+00:00,2023-05-03 08:36:09+00:00,271,NaT,False
4,333471,838023,2023-05-03 09:26:09+00:00,2023-05-03 12:23:17+00:00,271,NaT,False
...,...,...,...,...,...,...,...
151503,19772790,1574312,2023-06-11 22:17:56+00:00,2023-06-12 13:34:47+00:00,5372,NaT,False
151504,19837929,1574312,2023-06-12 22:23:11+00:00,2023-06-13 21:40:43+00:00,5372,NaT,False
151507,19963623,1575879,2023-06-15 10:31:01+00:00,2023-06-15 14:21:47+00:00,4426,NaT,False
151512,20052297,1576277,2023-06-17 11:35:32+00:00,2023-06-17 12:38:49+00:00,5520,NaT,False


* 각 행은 질문 세트 완료 한 건
* activation_at는 해당 유저가 이용할 수 있게 된 시점
* estimated_completed_at은 질문 세트를 완료했다고 추정한 시점

#### ⚠️ 0-4-9. 조건 검증 필요

In [61]:
first_questionset_activity = (
    target_questionsets[
        target_questionsets['estimated_completed_at'].notna()
    ]
    .groupby('user_id', as_index=False)
    .agg(
        first_questionset_opened_at=('opening_time', 'min'),
        first_questionset_completed_at=(
            'estimated_completed_at',
            'min',
        ),
    )
)

display(first_questionset_activity.head())

,user_id,first_questionset_opened_at,first_questionset_completed_at
0,838023,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00
1,838466,2023-05-02 22:41:59+00:00,2023-05-02 22:43:29+00:00
2,838642,2023-04-28 14:17:23+00:00,2023-04-28 14:19:21+00:00
3,839357,2023-04-29 16:50:47+00:00,2023-04-29 16:51:01+00:00
4,840293,2023-05-03 02:23:57+00:00,2023-05-03 02:25:41+00:00


In [62]:
first_questionset_activity = (
    target_questionsets[
        target_questionsets['estimated_completed_at'].notna()
    ]
    .sort_values(
        ['user_id', 'opening_time', 'questionset_id']
    )
    .drop_duplicates(
        subset='user_id',
        keep='first',
    )
    [
        [
            'user_id',
            'questionset_id',
            'opening_time',
            'estimated_completed_at',
        ]
    ]
    .rename(
        columns={
            'questionset_id': 'first_questionset_id',
            'opening_time': 'first_questionset_opened_at',
            'estimated_completed_at':
                'first_questionset_completed_at',
        }
    )
)

In [63]:
first_use_school_check = (
    first_questionset_activity
    .merge(
        user_activation[
            [
                'user_id',
                'school_id',
                'school_40_at',
            ]
        ],
        on='user_id',
        how='left',
        validate='one_to_one',
    )
)

In [112]:
first_use_school_check

,user_id,first_questionset_id,first_questionset_opened_at,first_questionset_completed_at,school_id,school_40_at,school_40_met_at_first_use
0,838023,116704,2023-04-29 10:02:26+00:00,2023-04-29 16:22:56+00:00,271,2023-04-28 03:43:18.503902+00:00,True
1,838466,305901,2023-05-02 22:41:59+00:00,2023-05-02 22:43:29+00:00,369,2023-05-02 22:44:27.267798+00:00,False
2,838642,102284,2023-04-28 14:17:23+00:00,2023-04-28 14:19:21+00:00,271,2023-04-28 03:43:18.503902+00:00,True
3,839357,133833,2023-04-29 16:50:47+00:00,2023-04-29 16:51:01+00:00,271,2023-04-28 03:43:18.503902+00:00,True
4,840293,315417,2023-05-03 02:23:57+00:00,2023-05-03 02:25:41+00:00,369,2023-05-02 22:44:27.267798+00:00,True
...,...,...,...,...,...,...,...
4807,1577437,20330127,2023-06-29 08:07:05+00:00,2023-06-29 08:08:45+00:00,4516,2023-05-02 15:17:04.497555+00:00,True
4808,1577440,20330495,2023-06-29 08:39:04+00:00,2023-06-29 09:04:32+00:00,4516,2023-05-02 15:17:04.497555+00:00,True
4809,1578095,20444430,2023-07-11 13:28:06+00:00,2023-07-11 13:35:33+00:00,369,2023-05-02 22:44:27.267798+00:00,True
4810,1579418,20586782,2023-07-25 07:14:57+00:00,2023-07-25 13:49:56+00:00,1719,2023-05-13 08:00:50.712386+00:00,True


In [64]:
first_use_school_check['school_40_met_at_first_use'] = (
    first_use_school_check['school_40_at'].notna()
    & (
        first_use_school_check['first_questionset_opened_at']
        >= first_use_school_check['school_40_at']
    )
)

In [65]:
school_condition_summary = (
    first_use_school_check[
        'school_40_met_at_first_use'
    ]
    .value_counts(dropna=False)
    .rename_axis('최초 이용 시 학교 40명 충족 여부')
    .reset_index(name='사용자 수')
)

school_condition_summary['사용자 비율(%)'] = (
    school_condition_summary['사용자 수']
    / school_condition_summary['사용자 수'].sum()
    * 100
).round(2)

display(school_condition_summary)

,최초 이용 시 학교 40명 충족 여부,사용자 수,사용자 비율(%)
0,True,4793,99.61
1,False,19,0.39


In [66]:
school_condition_exceptions = (
    first_use_school_check[
        ~first_use_school_check[
            'school_40_met_at_first_use'
        ]
    ]
    .sort_values('first_questionset_opened_at')
)

display(
    school_condition_exceptions[
        [
            'user_id',
            'school_id',
            'first_questionset_opened_at',
            'school_40_at',
        ]
    ]
)

,user_id,school_id,first_questionset_opened_at,school_40_at
443,869250,369,2023-05-02 21:37:17+00:00,2023-05-02 22:44:27.267798+00:00
506,872980,369,2023-05-02 21:55:57+00:00,2023-05-02 22:44:27.267798+00:00
507,872991,369,2023-05-02 22:05:57+00:00,2023-05-02 22:44:27.267798+00:00
455,871136,369,2023-05-02 22:06:24+00:00,2023-05-02 22:44:27.267798+00:00
416,865697,369,2023-05-02 22:21:15+00:00,2023-05-02 22:44:27.267798+00:00
8,840685,369,2023-05-02 22:21:43+00:00,2023-05-02 22:44:27.267798+00:00
461,871458,369,2023-05-02 22:24:48+00:00,2023-05-02 22:44:27.267798+00:00
12,841809,369,2023-05-02 22:24:53+00:00,2023-05-02 22:44:27.267798+00:00
447,869746,369,2023-05-02 22:24:55+00:00,2023-05-02 22:44:27.267798+00:00
435,868277,369,2023-05-02 22:25:07+00:00,2023-05-02 22:44:27.267798+00:00


In [67]:
school_condition_exceptions = (
    school_condition_exceptions
    .assign(
        exception_reason=np.where(
            school_condition_exceptions['school_40_at'].isna(),
            '학교 40명 미도달',
            '학교 40명 도달 이전 질문 이용',
        )
    )
)

display(
    school_condition_exceptions[
        'exception_reason'
    ].value_counts()
)

exception_reason
학교 40명 도달 이전 질문 이용    19
Name: count, dtype: int64

In [69]:
group_signup_order = (
    target_users
    .sort_values(
        ['group_id', 'signup_at', 'user_id']
    )
    .copy()
)

group_signup_order['group_signup_order'] = (
    group_signup_order
    .groupby('group_id')
    .cumcount()
    + 1
)

In [70]:
group_4_at = (
    group_signup_order[
        group_signup_order['group_signup_order'].eq(4)
    ]
    [
        [
            'group_id',
            'signup_at',
        ]
    ]
    .rename(
        columns={
            'signup_at': 'group_4_at',
        }
    )
    .reset_index(drop=True)
)

In [71]:
first_use_group_check = (
    first_use_school_check
    .merge(
        target_users[
            [
                'user_id',
                'group_id',
            ]
        ],
        on='user_id',
        how='left',
        validate='one_to_one',
    )
    .merge(
        group_4_at,
        on='group_id',
        how='left',
        validate='many_to_one',
    )
)

In [72]:
first_use_group_check['group_4_met_at_first_use'] = (
    first_use_group_check['group_4_at'].notna()
    & (
        first_use_group_check['first_questionset_opened_at']
        >= first_use_group_check['group_4_at']
    )
)

In [73]:
group_condition_summary = (
    first_use_group_check[
        'group_4_met_at_first_use'
    ]
    .value_counts(dropna=False)
    .rename_axis('최초 이용 시 그룹 4명 충족 여부')
    .reset_index(name='사용자 수')
)

group_condition_summary['사용자 비율(%)'] = (
    group_condition_summary['사용자 수']
    / group_condition_summary['사용자 수'].sum()
    * 100
).round(2)

display(group_condition_summary)

,최초 이용 시 그룹 4명 충족 여부,사용자 수,사용자 비율(%)
0,True,4183,86.93
1,False,629,13.07


In [74]:
first_use_friend_check = (
    first_use_group_check
    .merge(
        user_4th_friend_at,
        on='user_id',
        how='left',
        validate='one_to_one',
    )
)

In [75]:
first_use_friend_check['friend_4_met_at_first_use'] = (
    first_use_friend_check['friend_4_at'].notna()
    & (
        first_use_friend_check['first_questionset_opened_at']
        >= first_use_friend_check['friend_4_at']
    )
)

In [76]:
friend_condition_summary = (
    first_use_friend_check[
        'friend_4_met_at_first_use'
    ]
    .value_counts(dropna=False)
    .rename_axis('최초 이용 시 친구 4명 충족 여부')
    .reset_index(name='사용자 수')
)

friend_condition_summary['사용자 비율(%)'] = (
    friend_condition_summary['사용자 수']
    / friend_condition_summary['사용자 수'].sum()
    * 100
).round(2)

display(friend_condition_summary)

,최초 이용 시 친구 4명 충족 여부,사용자 수,사용자 비율(%)
0,False,4292,89.19
1,True,520,10.81


In [77]:
group_friend_count = pd.crosstab(
    first_use_friend_check['group_4_met_at_first_use'],
    first_use_friend_check['friend_4_met_at_first_use'],
)

group_friend_count.index.name = '그룹 4명 충족'
group_friend_count.columns.name = '친구 4명 충족'

display(group_friend_count)

친구 4명 충족,False,True
그룹 4명 충족,,
False,546,83
True,3746,437


In [78]:
group_friend_ratio = (
    group_friend_count
    / group_friend_count.to_numpy().sum()
    * 100
).round(2)

display(group_friend_ratio)

친구 4명 충족,False,True
그룹 4명 충족,,
False,11.35,1.72
True,77.85,9.08


In [79]:
both_unmet_users = (
    first_use_friend_check[
        ~first_use_friend_check['group_4_met_at_first_use']
        & ~first_use_friend_check['friend_4_met_at_first_use']
    ]
    .copy()
)

print(f'두 조건 모두 미충족 사용자: {len(both_unmet_users):,}명')

display(
    both_unmet_users[
        [
            'user_id',
            'school_id',
            'group_id',
            'first_questionset_opened_at',
            'group_4_at',
            'friend_4_at',
        ]
    ].head()
)

두 조건 모두 미충족 사용자: 546명


,user_id,school_id,group_id,first_questionset_opened_at,group_4_at,friend_4_at
3,839357,271,519,2023-04-29 16:50:47+00:00,2023-04-30 01:32:14.986496+00:00,2023-04-29 16:55:27+00:00
5,840473,369,915,2023-05-03 07:13:23+00:00,2023-05-03 08:09:03.087365+00:00,2023-05-04 13:50:41+00:00
6,840474,369,916,2023-05-03 00:43:30+00:00,2023-05-03 00:43:35.584947+00:00,2023-05-03 01:15:33+00:00
7,840512,369,930,2023-05-02 22:28:31+00:00,2023-05-03 05:51:24.351316+00:00,2023-05-03 06:20:08+00:00
9,840902,369,1058,2023-05-02 22:32:47+00:00,2023-05-03 02:10:35.148738+00:00,2023-05-02 22:33:04+00:00


In [80]:
school_group_order_check = (
    first_use_group_check[
        first_use_group_check['group_4_at'].notna()
        & first_use_group_check['school_40_at'].notna()
    ]
    .copy()
)

school_group_order_check['group_before_school'] = (
    school_group_order_check['group_4_at']
    < school_group_order_check['school_40_at']
)

In [81]:
display(
    school_group_order_check[
        'group_before_school'
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

group_before_school
False    87.43
True     12.57
Name: proportion, dtype: float64

In [82]:
group_before_school_users = (
    school_group_order_check[
        school_group_order_check['group_before_school']
    ]
    .copy()
)

group_before_school_users['first_use_timing'] = np.select(
    [
        (
            group_before_school_users['first_questionset_opened_at']
            < group_before_school_users['group_4_at']
        ),
        (
            group_before_school_users['first_questionset_opened_at']
            < group_before_school_users['school_40_at']
        ),
    ],
    [
        '그룹 4명 이전',
        '그룹 4명 이후 ~ 학교 40명 이전',
    ],
    default='학교 40명 이후',
)

display(
    group_before_school_users[
        'first_use_timing'
    ]
    .value_counts()
)

first_use_timing
학교 40명 이후    599
그룹 4명 이전       1
Name: count, dtype: int64

In [83]:
group_before_school_users['group_to_school_hours'] = (
    group_before_school_users['school_40_at']
    - group_before_school_users['group_4_at']
).dt.total_seconds() / 3600

In [84]:
display(
    group_before_school_users[
        'group_to_school_hours'
    ].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9]
    )
)

count    600.000000
mean      19.933130
std       32.078702
min        0.037833
25%        0.604760
50%        6.248909
75%       17.116090
90%       70.916984
max      143.830107
Name: group_to_school_hours, dtype: float64

In [85]:
pd.Series({
    '24시간 이상': (
        group_before_school_users['group_to_school_hours'] >= 24
    ).sum(),
    '72시간 이상': (
        group_before_school_users['group_to_school_hours'] >= 72
    ).sum(),
    '7일 이상': (
        group_before_school_users['group_to_school_hours'] >= 24 * 7
    ).sum(),
})

24시간 이상    144
72시간 이상     39
7일 이상        0
dtype: int64

In [86]:
group_window_summary = (
    group_before_school_users
    .groupby('group_id', as_index=False)
    .agg(
        school_id=('school_id', 'first'),
        group_4_at=('group_4_at', 'first'),
        school_40_at=('school_40_at', 'first'),
        first_questionset_opened_at=(
            'first_questionset_opened_at',
            'min',
        ),
    )
)

group_window_summary['group_to_school_hours'] = (
    group_window_summary['school_40_at']
    - group_window_summary['group_4_at']
).dt.total_seconds() / 3600

In [87]:
pd.Series({
    '비교 가능한 그룹 수': len(group_window_summary),

    '24시간 이상 간격 그룹': (
        group_window_summary['group_to_school_hours'] >= 24
    ).sum(),

    '72시간 이상 간격 그룹': (
        group_window_summary['group_to_school_hours'] >= 72
    ).sum(),

    '중간 구간 이용 그룹': (
        (
            group_window_summary['first_questionset_opened_at']
            >= group_window_summary['group_4_at']
        )
        & (
            group_window_summary['first_questionset_opened_at']
            < group_window_summary['school_40_at']
        )
    ).sum(),
})

비교 가능한 그룹 수      30
24시간 이상 간격 그룹     8
72시간 이상 간격 그룹     2
중간 구간 이용 그룹       0
dtype: int64

In [88]:
first_use_group_check['group_condition_reason'] = np.select(
    [
        first_use_group_check['group_4_at'].isna(),
        (
            first_use_group_check['first_questionset_opened_at']
            < first_use_group_check['group_4_at']
        ),
    ],
    [
        '현재 데이터에서 그룹 4명 미도달',
        '그룹 4명 도달 이전 이용',
    ],
    default='그룹 4명 도달 이후 이용',
)

display(
    first_use_group_check[
        'group_condition_reason'
    ]
    .value_counts()
)

group_condition_reason
그룹 4명 도달 이후 이용        4183
그룹 4명 도달 이전 이용         592
현재 데이터에서 그룹 4명 미도달      37
Name: count, dtype: int64

In [90]:
first_use_friend_check['any_condition_met_at_first_use'] = (
    first_use_friend_check['school_40_met_at_first_use']
    | first_use_friend_check['group_4_met_at_first_use']
    | first_use_friend_check['friend_4_met_at_first_use']
)

In [91]:
any_condition_summary = (
    first_use_friend_check[
        'any_condition_met_at_first_use'
    ]
    .value_counts(dropna=False)
    .rename_axis('세 조건 중 하나 이상 충족')
    .reset_index(name='사용자 수')
)

any_condition_summary['사용자 비율(%)'] = (
    any_condition_summary['사용자 수']
    / any_condition_summary['사용자 수'].sum()
    * 100
).round(2)

display(any_condition_summary)

,세 조건 중 하나 이상 충족,사용자 수,사용자 비율(%)
0,True,4801,99.77
1,False,11,0.23


In [92]:
print(
    '학교 조건만 충족률:',
    first_use_friend_check[
        'school_40_met_at_first_use'
    ].mean() * 100,
)

print(
    '세 조건 OR 충족률:',
    first_use_friend_check[
        'any_condition_met_at_first_use'
    ].mean() * 100,
)

학교 조건만 충족률: 99.60515378221115
세 조건 OR 충족률: 99.77140482128013


In [93]:
or_added_users = (
    first_use_friend_check[
        ~first_use_friend_check['school_40_met_at_first_use']
        & first_use_friend_check['any_condition_met_at_first_use']
    ]
    .copy()
)

display(
    or_added_users[
        [
            'user_id',
            'school_id',
            'group_id',
            'first_questionset_opened_at',
            'school_40_at',
            'group_4_at',
            'friend_4_at',
            'group_4_met_at_first_use',
            'friend_4_met_at_first_use',
        ]
    ]
)

,user_id,school_id,group_id,first_questionset_opened_at,school_40_at,group_4_at,friend_4_at,group_4_met_at_first_use,friend_4_met_at_first_use
1,838466,369,321,2023-05-02 22:41:59+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:02:13.179833+00:00,2023-05-01 14:10:16+00:00,False,True
8,840685,369,838,2023-05-02 22:21:43+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-02 22:42:11.069127+00:00,2023-05-02 12:57:04+00:00,False,True
356,858845,369,6506,2023-05-02 22:25:13+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:12:35.697785+00:00,2023-05-01 05:43:52+00:00,False,True
357,859087,369,6506,2023-05-02 22:26:35+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:12:35.697785+00:00,2023-05-02 09:35:20+00:00,False,True
434,868276,369,1379,2023-05-02 22:31:29+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:51:49.667380+00:00,2023-05-02 04:11:31+00:00,False,True
436,868425,369,916,2023-05-02 22:27:19+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:43:35.584947+00:00,2023-05-02 02:04:35+00:00,False,True
443,869250,369,321,2023-05-02 21:37:17+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 00:02:13.179833+00:00,2023-05-02 15:05:01+00:00,False,True
447,869746,369,1058,2023-05-02 22:24:55+00:00,2023-05-02 22:44:27.267798+00:00,2023-05-03 02:10:35.148738+00:00,2023-05-02 15:04:58+00:00,False,True


In [94]:
pd.crosstab(
    or_added_users['group_4_met_at_first_use'],
    or_added_users['friend_4_met_at_first_use'],
)

friend_4_met_at_first_use,True
group_4_met_at_first_use,
False,8


In [95]:
first_use_friend_check['all_conditions_met_at_first_use'] = (
    first_use_friend_check['school_40_met_at_first_use']
    & first_use_friend_check['group_4_met_at_first_use']
    & first_use_friend_check['friend_4_met_at_first_use']
)

In [96]:
and_condition_summary = (
    first_use_friend_check[
        'all_conditions_met_at_first_use'
    ]
    .value_counts(dropna=False)
    .rename_axis('세 조건 모두 충족')
    .reset_index(name='사용자 수')
)

and_condition_summary['사용자 비율(%)'] = (
    and_condition_summary['사용자 수']
    / and_condition_summary['사용자 수'].sum()
    * 100
).round(2)

display(and_condition_summary)

,세 조건 모두 충족,사용자 수,사용자 비율(%)
0,False,4375,90.92
1,True,437,9.08


In [97]:
first_use_friend_check['school_and_group_met'] = (
    first_use_friend_check['school_40_met_at_first_use']
    & first_use_friend_check['group_4_met_at_first_use']
)

In [98]:
school_group_and_summary = (
    first_use_friend_check[
        'school_and_group_met'
    ]
    .value_counts(dropna=False)
    .rename_axis('학교 40명·그룹 4명 모두 충족')
    .reset_index(name='사용자 수')
)

school_group_and_summary['사용자 비율(%)'] = (
    school_group_and_summary['사용자 수']
    / school_group_and_summary['사용자 수'].sum()
    * 100
).round(2)

display(school_group_and_summary)

,학교 40명·그룹 4명 모두 충족,사용자 수,사용자 비율(%)
0,True,4183,86.93
1,False,629,13.07


In [99]:
first_use_friend_check['school_or_group_met'] = (
    first_use_friend_check['school_40_met_at_first_use']
    | first_use_friend_check['group_4_met_at_first_use']
)

In [100]:
print(
    '학교 AND 그룹 충족률:',
    first_use_friend_check['school_and_group_met'].mean() * 100,
)

print(
    '학교 OR 그룹 충족률:',
    first_use_friend_check['school_or_group_met'].mean() * 100,
)

학교 AND 그룹 충족률: 86.92851205320034
학교 OR 그룹 충족률: 99.60515378221115


In [101]:
condition_check = (
    first_use_friend_check[
        [
            'user_id',
            'school_40_met_at_first_use',
            'group_4_met_at_first_use',
            'friend_4_met_at_first_use',
        ]
    ]
    .copy()
)

condition_check[
    [
        'school_40_met_at_first_use',
        'group_4_met_at_first_use',
        'friend_4_met_at_first_use',
    ]
] = condition_check[
    [
        'school_40_met_at_first_use',
        'group_4_met_at_first_use',
        'friend_4_met_at_first_use',
    ]
].fillna(False)

In [108]:
school_met = condition_check[
    'school_40_met_at_first_use'
]

group_met = condition_check[
    'group_4_met_at_first_use'
]

friend_met = condition_check[
    'friend_4_met_at_first_use'
]

condition_rules = {
    '1. 학교 AND 그룹 AND 친구': (
        school_met
        & group_met
        & friend_met
    ),

    '2. 학교 OR 그룹 OR 친구': (
        school_met
        | group_met
        | friend_met
    ),

    '3. 학교 AND 그룹': (
        school_met
        & group_met
    ),

    '4. 학교 OR 그룹': (
        school_met
        | group_met
    ),

    '5. 학교 AND 친구': (
        school_met
        & friend_met
    ),

    '6. 학교 OR 친구': (
        school_met
        | friend_met
    ),

    '7. 학교 단독': school_met,

    '8. 그룹 단독': group_met,

    '9. 친구 단독': friend_met,
}

In [109]:
total_users = condition_check['user_id'].nunique()

condition_comparison = pd.DataFrame([
    {
        '조건': condition_name,
        '전체 사용자 수': total_users,
        '충족 사용자 수': condition_result.sum(),
        '미충족 사용자 수': (~condition_result).sum(),
        '충족률(%)': condition_result.mean() * 100,
        '미충족률(%)': (~condition_result).mean() * 100,
    }
    for condition_name, condition_result
    in condition_rules.items()
])

condition_comparison[
    [
        '충족률(%)',
        '미충족률(%)',
    ]
] = condition_comparison[
    [
        '충족률(%)',
        '미충족률(%)',
    ]
].round(2)

display(condition_comparison)

,조건,전체 사용자 수,충족 사용자 수,미충족 사용자 수,충족률(%),미충족률(%)
0,1. 학교 AND 그룹 AND 친구,4812,437,4375,9.08,90.92
1,2. 학교 OR 그룹 OR 친구,4812,4801,11,99.77,0.23
2,3. 학교 AND 그룹,4812,4183,629,86.93,13.07
3,4. 학교 OR 그룹,4812,4793,19,99.61,0.39
4,5. 학교 AND 친구,4812,512,4300,10.64,89.36
5,6. 학교 OR 친구,4812,4801,11,99.77,0.23
6,7. 학교 단독,4812,4793,19,99.61,0.39
7,8. 그룹 단독,4812,4183,629,86.93,13.07
8,9. 친구 단독,4812,520,4292,10.81,89.19


## 1. 가설 설정

### 1-1. 친구 수에 따라 리텐션이 높을 것이다.

### 1-2. 활성화 조건 취득(학교 40명 & 친구 4명) 소요 시간이 짧을수록 리텐션이 높을 것이다.

### 1-3. 질문 완료 경험이 많을수록 리텐션이 높을 것이다.

### 1-4. 24시간 내 질문 완료 수가 많을수록 리텐션이 높을 것이다.

### 1-5. 질문 신고 수가 많을수록 리텐션이 높을 것이다.

### 1-6. 유저 신고 수가 많을수록 리텐션이 높을 것이다.

### 1-7. PING (질문에서 답변으로 선택된) 수가 많을수록 리텐션이 높을 것이다.